In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import numpy as np

from gerar_dados_vitimas import gerar_dataset_vitimas

In [ ]:
# Parâmetros do dataset (fixos para o trabalho)
N_VITIMAS = 6000
MEDIA_IDADE = 55
DESVIO_IDADE = 8
TIPO_ACIDENTE = "aereo"
NIVEL_RUIDO = 0.07
SEED_TREINO = 50
SEED_TESTE = 89

# Dataset de treino/validação
df_treino_val = gerar_dataset_vitimas(
    n_vitimas=N_VITIMAS,
    media_idade=MEDIA_IDADE,
    desvio_idade=DESVIO_IDADE,
    tipo_acidente=TIPO_ACIDENTE,
    nivel_ruido=NIVEL_RUIDO,
    seed=SEED_TREINO
)

In [ ]:
print("Formato do dataset:", df_treino_val.shape)
display(df_treino_val.head())
display(df_treino_val.describe())

In [ ]:
# variável alvo (target)
y = df_treino_val["sobr"]

# colunas proibidas pelo trabalho
colunas_proibidas = ["gcs", "avpu", "tri", "sobr"]

# features (entradas)
X = df_treino_val.drop(columns=colunas_proibidas)

print("Features utilizadas:")
print(list(X.columns))

print("\nFormato de X:", X.shape)
print("Formato de y:", y.shape)

## 2. Modelo CART
Primeiro realizamos a busca exaustiva para encontrar os melhores hiperparâmetros via Validação Cruzada.

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.tree import DecisionTreeRegressor
from itertools import product

kf = KFold(n_splits=5, shuffle=True, random_state=SEED_TREINO)

param_grid = {
    'max_depth': [3, 5, 10, 15],
    'min_samples_leaf': [5, 10, 20],
    'criterion': ['squared_error', 'friedman_mse']
}

keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in product(*values)]
resultados_lista = []

print(f"Iniciando busca exaustiva CART: {len(combinacoes)} combinações...\n")

for i, params in enumerate(combinacoes):
    model = DecisionTreeRegressor(**params, random_state=SEED_TREINO)
    cv_results = cross_validate(model, X, y, cv=kf, scoring="neg_mean_squared_error", return_train_score=True)
    
    mse_treino = -cv_results["train_score"]
    mse_valida = -cv_results["test_score"]
    
    resultados_lista.append({
        "max_depth": params['max_depth'],
        "min_samples_leaf": params['min_samples_leaf'],
        "criterion": params['criterion'],
        "mse_treino_medio": np.mean(mse_treino),
        "mse_valida_medio": np.mean(mse_valida),
        "var_treino": np.var(mse_treino),
        "var_valida": np.var(mse_valida),
        "diff_abs": abs(np.mean(mse_treino) - np.mean(mse_valida))
    })

df_resultados_cart = pd.DataFrame(resultados_lista).sort_values("mse_valida_medio")
display(df_resultados_cart.head(5))

melhor_cart_config = df_resultados_cart.iloc[0]
print(f"\nMELHOR CART ENCONTRADO: max_depth={melhor_cart_config['max_depth']}, min_samples_leaf={melhor_cart_config['min_samples_leaf']}, criterion={melhor_cart_res['criterion']}")

### 2.1 Retreino Final CART
Treinamos o modelo final com 100% dos dados usando os melhores parâmetros.

In [ ]:
modelo_final_cart = DecisionTreeRegressor(
    max_depth=melhor_cart_config['max_depth'],
    min_samples_leaf=melhor_cart_config['min_samples_leaf'],
    criterion=melhor_cart_config['criterion'],
    random_state=SEED_TREINO
)
modelo_final_cart.fit(X, y)
print("✅ Modelo CART final treinado com 100% dos dados!")

## 3. Modelo Rede Neural MLP
Agora repetimos o processo para a Rede Neural, incluindo a normalização dos dados.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

param_grid_mlp = {
    'hidden_layer_sizes': [(10,), (50,), (20, 10)], 
    'activation': ['tanh', 'relu'],                
    'learning_rate_init': [0.01, 0.001],           
    'solver': ['adam', 'lbfgs']                    
}

keys_mlp, values_mlp = zip(*param_grid_mlp.items())
combinacoes_mlp = [dict(zip(keys_mlp, v)) for v in product(*values_mlp)]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
resultados_mlp = []

print(f"Iniciando busca exaustiva MLP: {len(combinacoes_mlp)} combinações...\n")

for i, params in enumerate(combinacoes_mlp):
    model = MLPRegressor(**params, max_iter=1000, random_state=SEED_TREINO)
    cv_results = cross_validate(model, X_scaled, y, cv=kf, scoring="neg_mean_squared_error", return_train_score=True)
    
    mse_treino = -cv_results["train_score"]
    mse_valida = -cv_results["test_score"]
    
    resultados_mlp.append({
        "config_str": str(params),
        "params_dict": params,
        "mse_treino_medio": np.mean(mse_treino),
        "mse_valida_medio": np.mean(mse_valida),
        "var_treino": np.var(mse_treino),
        "var_valida": np.var(mse_valida),
        "diff_abs": abs(np.mean(mse_treino) - np.mean(mse_valida))
    })

df_resultados_mlp = pd.DataFrame(resultados_mlp).sort_values("mse_valida_medio")
display(df_resultados_mlp.head(5))

melhor_mlp_config = df_resultados_mlp.iloc[0]
print(f"\nMELHOR MLP ENCONTRADA: {melhor_mlp_config['config_str']}")

### 3.1 Retreino Final MLP
Treinamos o modelo final de Rede Neural com 100% dos dados (normalizados).

In [ ]:
modelo_final_mlp = MLPRegressor(**melhor_mlp_config['params_dict'], max_iter=1000, random_state=SEED_TREINO)
modelo_final_mlp.fit(X_scaled, y)
print("✅ Modelo MLP final treinado com 100% dos dados!")